# Step 5: Classical to Quantum — The Heisenberg Model and Exact Diagonalisation

Every spin in notebooks 01–04 was a classical variable — a number $\pm 1$ or an angle $\theta$. Here we make the jump to quantum mechanics. Each spin becomes a quantum operator with non-commuting components, the Hilbert space grows exponentially with system size, and the ground state is a superposition over all $2^N$ classical configurations.

We use the 1D Heisenberg chain as a concrete model to introduce **exact diagonalisation (ED)**: building the Hamiltonian as a sparse matrix and finding its eigenvalues numerically. All the ED machinery developed here carries directly into notebooks 06–09.

**What you will do:**
- Define single-site spin-$1/2$ operators and verify the commutation relations
- Encode the many-body basis as binary integers and build the sparse Hamiltonian
- Diagonalise the Heisenberg chain and verify the ground state energies
- Track how the ground state energy per site converges to the Bethe ansatz result
- Measure spin-spin correlations and the structure factor
- Observe that the energy gap vanishes with system size — the chain is gapless

**Prerequisites:** Notebooks 01–04.

## 5.1  Spin-$1/2$ Operators

A classical spin $s_i \in \{+1,-1\}$ is replaced by a quantum operator $\hat{\vec{S}}_i = (\hat{S}^x_i, \hat{S}^y_i, \hat{S}^z_i)$ acting on the two-dimensional local Hilbert space spanned by $|{\uparrow}\rangle$ and $|{\downarrow}\rangle$. In matrix form:

$$\hat{S}^x = \frac{1}{2}\begin{pmatrix}0&1\\1&0\end{pmatrix}, \quad
\hat{S}^y = \frac{1}{2}\begin{pmatrix}0&-i\\i&0\end{pmatrix}, \quad
\hat{S}^z = \frac{1}{2}\begin{pmatrix}1&0\\0&-1\end{pmatrix}$$

The raising and lowering operators $\hat{S}^\pm = \hat{S}^x \pm i\hat{S}^y$ flip a spin:
$\hat{S}^+|{\downarrow}\rangle = |{\uparrow}\rangle$, $\quad\hat{S}^-|{\uparrow}\rangle = |{\downarrow}\rangle$.

The commutation relations $[\hat{S}^x, \hat{S}^y] = i\hat{S}^z$ (and cyclic) are what distinguish quantum spins from classical ones — the three components cannot be simultaneously specified.

In [ ]:
import numpy as np
from scipy.sparse import lil_matrix, csr_matrix
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt

# Single-site spin-1/2 matrices
Sx = 0.5 * np.array([[0,  1 ], [1,  0 ]], dtype=complex)
Sy = 0.5 * np.array([[0, -1j], [1j, 0 ]], dtype=complex)
Sz = 0.5 * np.array([[1,  0 ], [0, -1 ]], dtype=complex)
Sp = np.array([[0, 1], [0, 0]], dtype=complex)   # S+ = Sx + i*Sy
Sm = np.array([[0, 0], [1, 0]], dtype=complex)   # S- = Sx - i*Sy

# Verify commutation relations: [Sx, Sy] = i*Sz,  [Sy, Sz] = i*Sx,  [Sz, Sx] = i*Sy
print("[Sx, Sy] = i*Sz :", np.allclose(Sx @ Sy - Sy @ Sx,  1j * Sz))
print("[Sy, Sz] = i*Sx :", np.allclose(Sy @ Sz - Sz @ Sy,  1j * Sx))
print("[Sz, Sx] = i*Sy :", np.allclose(Sz @ Sx - Sx @ Sz,  1j * Sy))

# Verify raising/lowering action on basis states
up   = np.array([1, 0])   # |up>
down = np.array([0, 1])   # |down>
print()
print("S+ |down> = |up>  :", np.allclose(Sp @ down, up))
print("S- |up>   = |down>:", np.allclose(Sm @ up,   down))
print("S+ |up>   = 0     :", np.allclose(Sp @ up,   0))
print("S- |down> = 0     :", np.allclose(Sm @ down, 0))

## 5.2  The Many-Body Hilbert Space

For a chain of $N$ spin-$1/2$ sites, the Hilbert space is the tensor product of $N$ two-dimensional spaces, with dimension $2^N$. We label the $2^N$ basis states by integers $0, 1, \ldots, 2^N - 1$, where bit $i$ of the integer encodes the spin at site $i$:

$$\text{bit } i = 1 \; \Leftrightarrow \; |{\uparrow}\rangle \text{ at site } i, \qquad \text{bit } i = 0 \; \Leftrightarrow \; |{\downarrow}\rangle \text{ at site } i$$

This encoding makes it cheap to read the spin at any site with a bitwise operation: `(state >> i) & 1`, and to flip spins by XOR: `state ^ (1 << i)`.

In [ ]:
def state_label(state, N):
    """Return the ket string for a basis state integer."""
    spins = ''.join('\u2191' if (state >> i) & 1 else '\u2193' for i in range(N))
    return f"|{spins}\u27e9"

N_demo = 3
dim    = 2 ** N_demo
print(f"N = {N_demo}:  Hilbert space dimension = 2^{N_demo} = {dim}")
print()
print(f"{'Integer':>8}  {'Binary':>6}  {'Ket':>10}  {'Sz_tot':>8}")
print("-" * 40)
for s in range(dim):
    sz_tot = sum((s >> i) & 1 for i in range(N_demo)) - N_demo / 2
    print(f"{s:>8}  {s:>06b}  {state_label(s, N_demo):>10}  {sz_tot:>8.1f}")

print()
print("Hilbert space dimensions for increasing N:")
for N in [4, 8, 12, 16, 20, 30, 50]:
    print(f"  N = {N:>2}: dim = 2^{N} = {2**N:.2e}")

## 5.3  Quantum Fluctuations

Before building the matrix, it is worth understanding *why* the quantum problem is fundamentally harder than the classical one.

In the classical Ising ferromagnet ($J > 0$), the ground states $|\uparrow\uparrow\cdots\uparrow\rangle$ and $|\downarrow\downarrow\cdots\downarrow\rangle$ are exact eigenstates of the Hamiltonian. The system sits in one of them and nothing happens at $T = 0$.

In the classical antiferromagnet, the analogous ground state candidate is the **Néel state** — the perfectly alternating configuration $|\uparrow\downarrow\uparrow\downarrow\cdots\rangle$ in which every spin is anti-aligned with its neighbours. For the *classical* Ising antiferromagnet, this is indeed the ground state.

For the *quantum* Heisenberg antiferromagnet, the Néel state is **not** an eigenstate. To see why, apply the Hamiltonian to $|\uparrow\downarrow\uparrow\downarrow\cdots\rangle$. The $\hat{S}^+_i\hat{S}^-_j$ term on any anti-aligned bond creates a new configuration: $|\cdots\downarrow\uparrow\cdots\rangle$. The Hamiltonian immediately mixes the Néel state with other configurations, so it cannot be a stationary state.

The true ground state is therefore a **quantum superposition** — a linear combination of exponentially many classical configurations. Even at $T = 0$, the system fluctuates between them. These are **quantum fluctuations**: they have nothing to do with thermal noise and are always present. They arise purely from the non-commutativity of $\hat{S}^x$, $\hat{S}^y$, $\hat{S}^z$.

Two consequences worth noting:

- The 1D quantum Heisenberg antiferromagnet has **no long-range Néel order** even at $T = 0$. Quantum fluctuations destroy the order that would exist in the classical model. Its ground state is a disordered singlet — a quantum spin liquid.
- In 2D, there is long-range order, but the staggered magnetisation is reduced from its classical value by roughly 40% due to quantum fluctuations. This reduction is measurable in neutron scattering experiments.

This is the key qualitative change when going from classical to quantum spins. The computational consequence is that we can no longer describe the ground state by specifying one spin configuration: we need all $2^N$ amplitudes.

## 5.3  Building the Heisenberg Hamiltonian

The 1D Heisenberg Hamiltonian is:

$$\hat{H} = -J \sum_{\langle i,j \rangle} \hat{\vec{S}}_i \cdot \hat{\vec{S}}_j
= -J \sum_{\langle i,j \rangle} \left[\hat{S}^z_i \hat{S}^z_j
+ \frac{1}{2}\bigl(\hat{S}^+_i \hat{S}^-_j + \hat{S}^-_i \hat{S}^+_j\bigr)\right]$$

We sign-convention: $J > 0$ ferromagnet, $J < 0$ antiferromagnet.

For each nearest-neighbour bond $(i, j)$ and each basis state $|s\rangle$, two rules apply:

**Diagonal ($\hat{S}^z_i \hat{S}^z_j$):** add $-J \cdot s^z_i \cdot s^z_j$ to $H_{ss}$, where $s^z_i = \pm 1/2$.

**Off-diagonal ($\hat{S}^\pm_i \hat{S}^\mp_j$):** if spins $i$ and $j$ are anti-aligned, flip both to get $|s'\rangle$ and add $-J/2$ to $H_{s's}$.

The matrix is sparse: at most $N$ off-diagonal non-zero entries per row. We store it in CSR format.

In [ ]:
def build_heisenberg(N, J=1.0, pbc=True):
    """
    Build the 1D Heisenberg Hamiltonian as a sparse CSR matrix.
    H = -J sum_{<i,j>} [Sz_i Sz_j + (S+_i S-_j + S-_i S+_j) / 2]

    Basis: integers 0..2^N-1, bit i = 1 means spin-up at site i.
    J > 0: ferromagnet.  J < 0: antiferromagnet.
    pbc: periodic boundary conditions.
    """
    dim    = 2 ** N
    bonds  = [(i, (i + 1) % N) for i in range(N if pbc else N - 1)]
    states = np.arange(dim)

    rows, cols, vals = [], [], []

    for (i, j) in bonds:
        bi   = (states >> i) & 1          # bit at site i: 1=up, 0=down
        bj   = (states >> j) & 1
        sz_i = bi.astype(float) - 0.5     # +0.5 or -0.5
        sz_j = bj.astype(float) - 0.5

        # Diagonal: -J * Sz_i * Sz_j
        rows.extend(states.tolist())
        cols.extend(states.tolist())
        vals.extend((-J * sz_i * sz_j).tolist())

        # Off-diagonal: flip both spins when anti-aligned
        mask    = bi != bj
        s_anti  = states[mask]
        s_flip  = s_anti ^ (1 << i) ^ (1 << j)
        n_anti  = int(mask.sum())

        rows.extend(s_flip.tolist())
        cols.extend(s_anti.tolist())
        vals.extend([-J * 0.5] * n_anti)

    H = csr_matrix((vals, (rows, cols)), shape=(dim, dim), dtype=float)
    return H


# ── Verify against known analytical results ───────────────────────────────────
# N=2, J=-1 (AFM): singlet E0=-3/4, triplet E=+1/4
H2 = build_heisenberg(N=2, J=-1.0, pbc=False)   # single bond
evals_2 = np.linalg.eigh(H2.toarray())[0]
print("N=2, J=-1 eigenvalues:", np.sort(evals_2))
print("  Expected: [-0.75, 0.25, 0.25, 0.25]")
print()

# N=4, J=1 (FM): E0 = -J*N/4 = -1.0
H4_fm = build_heisenberg(N=4, J=1.0, pbc=True)
e0_fm = eigsh(H4_fm, k=1, which='SA', return_eigenvectors=False)[0]
print(f"N=4, J=+1 (FM): E0 = {e0_fm:.6f}  (expected {-4*1/4:.6f})")

# N=4, J=-1 (AFM): sparse solve
H4_afm = build_heisenberg(N=4, J=-1.0, pbc=True)
e0_afm = eigsh(H4_afm, k=1, which='SA', return_eigenvectors=False)[0]
print(f"N=4, J=-1 (AFM): E0/N = {e0_afm/4:.6f}")

## 5.4  Diagonalisation

With the sparse Hamiltonian in hand, we find eigenvalues in two ways:

- **Full spectrum** (`numpy.linalg.eigh` on the dense matrix): all $2^N$ eigenvalues. Memory cost $O(4^N)$; practical up to $N \sim 16$.
- **Ground state only** (`scipy.sparse.linalg.eigsh` with Lanczos): iteratively finds the few lowest eigenvalues using only matrix-vector products. Practical up to $N \sim 30$–40.

The Lanczos algorithm builds a Krylov subspace $\{|v\rangle, \hat{H}|v\rangle, \hat{H}^2|v\rangle, \ldots\}$ from a random start vector and finds extreme eigenvalues within it. After $k \sim 100$ iterations it typically converges the ground state energy to machine precision.

Here we use `eigsh` throughout since it scales to larger systems. We request `k=3` eigenvalues to access both the ground state and the first excited states.

In [ ]:
J_AFM = -1.0   # antiferromagnetic coupling
J_FM  =  1.0   # ferromagnetic coupling

N_demo2 = 8
H_afm   = build_heisenberg(N_demo2, J=J_AFM, pbc=True)

# Lanczos: 4 lowest eigenvalues
evals, evecs = eigsh(H_afm, k=4, which='SA')
evals = np.sort(evals)

print(f"N = {N_demo2}, J = {J_AFM}, PBC")
print(f"  Hilbert space dimension: {2**N_demo2}")
print(f"  4 lowest eigenvalues: {evals}")
print(f"  E0/N = {evals[0]/N_demo2:.6f}")
print(f"  Gap (E1-E0) = {evals[1] - evals[0]:.6f}")
print()

# Cross-check with full diagonalisation (affordable for N=8)
evals_full = np.sort(np.linalg.eigh(H_afm.toarray())[0])
print(f"  Full diag E0   = {evals_full[0]:.10f}")
print(f"  Lanczos  E0   = {evals[0]:.10f}")
print(f"  Agreement: {np.isclose(evals_full[0], evals[0])}")

## 5.5  Ground State Energy and Finite-Size Convergence

For the 1D antiferromagnetic Heisenberg chain (open or periodic), the ground state energy per site converges to the **Bethe ansatz** exact result as $N \to \infty$:

$$\frac{E_0}{N} \xrightarrow{N \to \infty} \frac{1}{4} - \ln 2 \approx -0.4431 \quad (J=-1)$$

This is one of the few exact results in quantum many-body physics. Tracking the convergence with ED is both a validation of the code and a demonstration of finite-size effects in quantum systems — directly analogous to the finite-size rounding we saw in notebooks 01–02.

In [ ]:
E_BETHE = 0.25 - np.log(2)   # exact Bethe ansatz: 1/4 - ln(2) ≈ -0.4431

N_values = [4, 6, 8, 10, 12, 14, 16, 18, 20]
e0_per_site = []

for N in N_values:
    H = build_heisenberg(N, J=J_AFM, pbc=True)
    e0 = eigsh(H, k=1, which='SA', return_eigenvectors=False)[0]
    e0_per_site.append(e0 / N)
    print(f"N = {N:>2}: E0/N = {e0/N:.8f}  (error = {abs(e0/N - E_BETHE):.2e})")

print(f"\nBethe ansatz (N→∞): E0/N = {E_BETHE:.8f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: E0/N vs N
axes[0].plot(N_values, e0_per_site, 'o-', ms=6, lw=1.5, color='steelblue',
             label="ED (PBC)")
axes[0].axhline(E_BETHE, color='k', ls='--', lw=1.2,
                label=f"Bethe ansatz = {E_BETHE:.4f}")
axes[0].set_xlabel("System size $N$", fontsize=13)
axes[0].set_ylabel("$E_0 / N$", fontsize=13)
axes[0].set_title("Ground state energy per site", fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Right: error vs 1/N^2 (leading finite-size correction)
inv_N2 = [1 / N**2 for N in N_values]
errors = [abs(e - E_BETHE) for e in e0_per_site]
axes[1].loglog(inv_N2, errors, 'o-', ms=6, lw=1.5, color='tomato')
# Guide for the eye: ~1/N^2 slope
x_fit = np.array(inv_N2)
axes[1].loglog(x_fit, 0.3 * x_fit, 'k--', lw=1, label=r"$\propto 1/N^2$")
axes[1].set_xlabel(r"$1/N^2$", fontsize=13)
axes[1].set_ylabel(r"$|E_0/N - E_{\rm Bethe}|$", fontsize=13)
axes[1].set_title("Finite-size error", fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3, which='both')

plt.tight_layout()
plt.show()

## 5.6  Spin Correlations and the Structure Factor

The ground state wavefunction encodes spatial correlations. The spin-spin correlation function:

$$C(r) = \langle E_0 | \hat{S}^z_0 \hat{S}^z_r | E_0 \rangle$$

Since $\hat{S}^z_i$ is diagonal in the computational basis — $\hat{S}^z_i|s\rangle = s^z_i|s\rangle$ — the expectation value reduces to a weighted sum over basis states:

$$C(r) = \sum_{s=0}^{2^N-1} |\psi_s|^2 \, s^z_0(s) \cdot s^z_r(s)$$

where $\psi_s$ is the amplitude of basis state $s$ in the ground state.

For the antiferromagnet, $C(r)$ alternates in sign: $C(r) \sim (-1)^r / r$. The **structure factor**:

$$S(k) = \frac{1}{N} \sum_{i,j} e^{ik(i-j)} \langle \hat{S}^z_i \hat{S}^z_j \rangle$$

peaks at $k = \pi$, revealing the antiferromagnetic ordering wavevector.

In [ ]:
N_corr = 14
H_corr = build_heisenberg(N_corr, J=J_AFM, pbc=True)
_, evecs_corr = eigsh(H_corr, k=1, which='SA')
psi0 = evecs_corr[:, 0]   # ground state vector, shape (2^N,)

states_arr = np.arange(2 ** N_corr)
prob        = np.abs(psi0) ** 2   # |<s|psi0>|^2

def sz_at(states, site, N):
    """Sz eigenvalue at given site for each basis state."""
    return ((states >> site) & 1).astype(float) - 0.5

def correlator_zz(psi0, N, site_i, site_j):
    """<psi0 | Sz_i Sz_j | psi0> using diagonal representation."""
    states = np.arange(2 ** N)
    sz_i   = sz_at(states, site_i, N)
    sz_j   = sz_at(states, site_j, N)
    return float(np.sum(np.abs(psi0) ** 2 * sz_i * sz_j))

# C(r) = <Sz_0 Sz_r> for r = 0..N/2
r_values = np.arange(N_corr // 2 + 1)
Czz      = np.array([correlator_zz(psi0, N_corr, 0, r) for r in r_values])

# Structure factor S(k)
k_values  = 2 * np.pi * np.arange(N_corr) / N_corr
# Full correlation matrix for S(k)
Czz_full  = np.array([[correlator_zz(psi0, N_corr, i, j)
                        for j in range(N_corr)] for i in range(N_corr)])
Sk = np.array([
    (1 / N_corr) * np.sum(Czz_full * np.exp(1j * k * (np.arange(N_corr)[:, None]
                                                      - np.arange(N_corr)[None, :])))
    for k in k_values
]).real

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(r_values, Czz, 'o-', ms=6, lw=1.5, color='steelblue')
axes[0].axhline(0, color='k', lw=0.8, ls=':')
axes[0].set_xlabel("Separation $r$", fontsize=13)
axes[0].set_ylabel(r"$\langle \hat{S}^z_0 \hat{S}^z_r \rangle$", fontsize=13)
axes[0].set_title(f"Spin correlation function  ($N={N_corr}$, AFM)", fontsize=13)
axes[0].grid(alpha=0.3)

axes[1].plot(k_values / np.pi, Sk, 'o-', ms=6, lw=1.5, color='tomato')
axes[1].axvline(1.0, color='k', ls='--', lw=1.2, label=r"$k = \pi$")
axes[1].set_xlabel(r"$k / \pi$", fontsize=13)
axes[1].set_ylabel(r"$S(k)$", fontsize=13)
axes[1].set_title("Structure factor", fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Peak of S(k) at k/pi = {k_values[np.argmax(Sk)]/np.pi:.3f}  (expected 1.0)")

## 5.7  The Energy Gap

The **spectral gap** $\Delta = E_1 - E_0$ is the energy cost of the lowest excitation above the ground state. It distinguishes two qualitatively different phases:

- **Gapped:** $\Delta > 0$ in the thermodynamic limit $N \to \infty$. Low-energy excitations are massive.
- **Gapless:** $\Delta \to 0$ as $N \to \infty$. The spectrum is continuous down to zero energy.

The 1D spin-$1/2$ Heisenberg antiferromagnet is **gapless** — it has a linearly dispersing spectrum (magnons) with no gap. The finite-size gap scales as $\Delta(N) \sim 1/N$, and a log-log plot of $\Delta$ vs $N$ should give a slope of $-1$.

This is in contrast to the spin-1 Heisenberg chain, which has a finite gap (the **Haldane gap**) — a purely quantum mechanical phenomenon.

In [ ]:
N_gap_values = [6, 8, 10, 12, 14, 16, 18, 20]
gaps = []

for N in N_gap_values:
    H = build_heisenberg(N, J=J_AFM, pbc=True)
    # Request 4 eigenvalues to safely find E1 (ground state may be degenerate)
    evals = np.sort(eigsh(H, k=4, which='SA', return_eigenvectors=False))
    gap   = evals[1] - evals[0]
    gaps.append(gap)
    print(f"N = {N:>2}:  E0 = {evals[0]:.6f},  E1 = {evals[1]:.6f},  gap = {gap:.6f}")

# Fit slope on log-log scale
log_N   = np.log(N_gap_values)
log_gap = np.log(gaps)
slope, intercept = np.polyfit(log_N, log_gap, 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

N_fit = np.linspace(min(N_gap_values) * 0.9, max(N_gap_values) * 1.1, 100)

axes[0].plot(N_gap_values, gaps, 'o', ms=7, color='steelblue', label="ED gap")
axes[0].plot(N_fit, np.exp(intercept) * N_fit ** slope, 'k--', lw=1.5,
             label=f"Fit: $\\Delta \\sim N^{{{slope:.2f}}}$")
axes[0].set_xlabel("System size $N$", fontsize=13)
axes[0].set_ylabel(r"Gap $\Delta = E_1 - E_0$", fontsize=13)
axes[0].set_title("Energy gap vs system size", fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

axes[1].loglog(N_gap_values, gaps, 'o', ms=7, color='steelblue', label="ED gap")
axes[1].loglog(N_fit, np.exp(intercept) * N_fit ** slope, 'k--', lw=1.5,
               label=f"Fit slope = {slope:.2f}  (expected $-1$)")
axes[1].set_xlabel("System size $N$", fontsize=13)
axes[1].set_ylabel(r"Gap $\Delta$", fontsize=13)
axes[1].set_title("Log-log: gap $\\sim N^{-1}$ (gapless chain)", fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print(f"\nFitted exponent: {slope:.3f}  (expected -1 for gapless chain)")

## 5.8  Symmetries, Block Diagonalisation, and Limitations of ED

### Exploiting symmetries

The Heisenberg Hamiltonian commutes with the total $z$-magnetisation $\hat{S}^z_\text{tot} = \sum_i \hat{S}^z_i$. This means $\hat{H}$ is **block-diagonal** in sectors of fixed $S^z_\text{tot}$, and each sector can be diagonalised independently.

The sector with $S^z_\text{tot} = 0$ (equal numbers of up and down spins) has dimension $\binom{N}{N/2}$. For $N = 24$ this is $2\,704\,156$ instead of $2^{24} = 16\,777\,216$ — a factor of six reduction in matrix size, and a factor of thirty-six in memory. The ground state of the antiferromagnet always lies in the $S^z_\text{tot} = 0$ sector.

Additional symmetries — spatial translation (for PBC chains), parity (reflection), and spin inversion — can each roughly halve the effective dimension again, pushing the accessible system size from $N \sim 20$ to $N \sim 36$–$40$ for the Heisenberg chain. Exercise 2 asks you to implement the $S^z_\text{tot}$ block diagonalisation explicitly.

### Limitations of ED

The exponential growth of the Hilbert space is the fundamental wall. In practice, ED reaches:

- $N \sim 40$ sites for the 1D Heisenberg chain with all symmetries used.
- $N \sim 20$–30 for models without conservation laws.
- Much smaller for 2D systems: a $6 \times 6$ square lattice has $N = 36$ sites, dimension $\sim 7 \times 10^{10}$ — already out of reach.

Beyond these sizes, we need tensor networks (notebooks 09 and 10) or quantum Monte Carlo. ED remains essential as a benchmark: any new method must reproduce ED results for the accessible system sizes. When DMRG in notebook 10 gives an energy within $10^{-8}$ of the ED result, that agreement is the validation.

## Exercises

1. **Ferromagnet vs antiferromagnet.** Run `build_heisenberg` with $J = +1$ (FM) for $N = 4, 6, 8$. The FM ground state energy per site is $-J/4 = -0.25$ exactly (the fully polarised state). Confirm this and verify that the FM ground state is $(2S_{\rm tot}+1)$-fold degenerate (for $S_{\rm tot} = N/2$ and PBC, this is $N+1$).

2. **$S^z_{\rm tot}$ conservation.** The Heisenberg Hamiltonian commutes with $\hat{S}^z_{\rm tot}$. Verify this numerically for $N = 6$: compute $[H, S^z_{\rm tot}]$ as dense matrices and check it is zero. Then restrict the Hilbert space to the $S^z_{\rm tot} = 0$ sector (states with exactly $N/2$ up-spins) and rebuild and diagonalise $H$ in that sector. Compare the lowest eigenvalues to those from the full matrix.

3. **Open boundary conditions.** Repeat the ground state energy convergence (section 5.5) with `pbc=False`. The finite-size corrections are different (boundary effects break translation invariance) and the convergence is slower. Compare the two convergence curves.

4. **Spin-1 Heisenberg chain.** The spin-1 Heisenberg chain has a finite gap (the Haldane gap $\Delta \approx 0.41J$). Extend `build_heisenberg` to spin-1 by replacing the spin-$1/2$ operators with their spin-1 analogues (3×3 matrices) and compute the gap for $N = 4, 6, 8, 10$. Does it appear to converge to a finite value rather than zero?